# Dauer nociception simulation - FLIP
- Refer to __ Pechuck et. al. 2022<br>
- 230724 new ID ver.

## Searching crucial dimorphic connection using `'flip'` approach

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile as tif
import networkx as nx
import os
import copy
import pickle
import itertools
import scipy

from tqdm import tqdm
import scipy.stats as stats
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.ticker import FormatStrFormatter
from scipy.stats import linregress
import matplotlib.colors as colors
from matplotlib import cm

## 1) ID and types

In [ ]:
# common type

com_fname = r'your_directory\230724_neurontype_new.csv'    # new common id (230724 -)
com_id_df = pd.read_csv(com_fname)
com_id = np.array(com_id_df[['Neuron','Common_id']])
    # Cook vs. Witvliet
com_type = np.array(com_id_df['Cook_typecode'])    # change


In [ ]:
# get index (230724 ver.)
# ========================================================================================================================
com_L_idx = [0,2,4,6,8,10,12,14,16,18,20,22,23,25,27,28,30,32,34,36,38,40,42,44,46,48,50,52,54,56,58,59,60,62,64,66,68,70,\
             72,74,75,76,78,80,82,84,86,88,90,92,94,96,98,100,102,104,106,108,109,110,112,114,116,117,119,121,122,124,126,\
             127,128,130,132,134,136,137,139,140,142,144,146,148,150,152,154,156,158,160,162,164,166,168,170,172,174,176,\
             178,180,181,182,183,184,185,186,187,196,197,198,199,200,201,202,203,212,214,216,218,220]
com_R_idx = [1,3,5,7,9,11,13,15,17,19,21,22,24,26,27,29,31,33,35,37,39,41,43,45,47,49,51,53,55,57,58,59,61,63,65,67,69,71,\
             73,74,75,77,79,81,83,85,87,89,91,93,95,97,99,101,103,105,107,108,109,111,113,115,116,118,120,121,123,125,126,\
             127,129,131,133,135,136,138,139,141,143,145,147,149,151,153,155,157,159,161,163,165,167,169,171,173,175,177,\
             179,188,189,190,191,192,193,194,195,204,205,206,207,208,209,210,211,213,215,217,219,221]

com_D_idx = [22,27,58,59,70,71,74,75,80,81,86,87,94,95,108,109,116,121,126,127,130,131,136,146,147,152,153,\
             156,157,160,161,164,165,168,169,176,177,180,181,182,183,184,185,186,187,188,189,190,191,192,\
             193,194,195,212,213,216,217]
com_V_idx = [22,27,58,59,72,73,74,75,84,85,90,91,96,97,108,109,116,121,126,127,134,135,139,148,149,154,155,\
             158,159,162,163,166,167,170,171,178,179,196,197,198,199,200,201,202,203,204,205,206,207,208,\
             209,210,211,214,215,220,221]
# ========================================================================================================================
com_L_idx = np.array(com_L_idx); com_R_idx = np.array(com_R_idx)
com_D_idx = np.array(com_D_idx); com_V_idx = np.array(com_V_idx)

# L/R/D/V id
LR_id = ['ADA','ADE','ADF','ADL','AFD','AIA','AIB','AIM','AIN','AIY','AIZ','ALA','ALM','ALN','AQR','ASE','ASG',\
         'ASH','ASI','ASJ','ASK','AUA','AVA','AVB','AVD','AVE','AVF','AVH','AVJ','AVK','AVL','AVM','AWA','AWB',\
         'AWC','BAG','BDU','CEPD','CEPV','DVA','DVC','FLP','HSN','IL1D','IL1','IL1V','IL2D','IL2','IL2V','OLL',\
         'OLQD','OLQV','PLN','PVC','PVN','PVP','PVQ','PVR','PVT','RIA','RIB','RIC','RID','RIF','RIG','RIH',\
         'RIM','RIP','RIR','RIS','RIV','RMDD','RMD','RMDV','RMED','RME','RMEV','RMF','RMG','RMH','SAAD','SAAV',\
         'SDQ','SIAD','SIAV','SIBD','SIBV','SMBD','SMBV','SMDD','SMDV','URAD','URAV','URB','URX','URYD','URYV']

DV_id = ['ALA','AQR','AVL','AVM','CEP L','CEP R','DVA','DVC','IL1 L','IL1 R','IL2 L','IL2 R','OLQ L','OLQ R',\
         'PVR','PVT','RID','RIH','RIR','RIS','RMD L','RMD R','RME','SAA L','SAA R','SIA L','SIA R',\
         'SIB L','SIB R','SMB L','SMB R','SMD L','SMD R','URA L','URA R','URY L','URY R']


## 2) Connectivity matrices

In [ ]:
# normalized matrix
    # from matrix by HSY (230724 compilation ver.)

norm180_comp = np.load(r"your_directory\norm180_11_compilation_230724.npy", allow_pickle=True); print(norm180_comp.shape)
    # fixed 230724
norm222_comp = np.load(r"your_directory\norm222_11_compilation_230724.npy", allow_pickle=True); print(norm222_comp.shape,"\n")
    # fixed 230724

# binarize
norm180_comp_bi = np.copy(norm180_comp); norm180_comp_bi[norm180_comp_bi!=0] = 1
norm222_comp_bi = np.copy(norm222_comp); norm222_comp_bi[norm222_comp_bi!=0] = 1


# others (original) - 230724 ver. (gap junctions debugged on 240413)
n2u_chem_mat = np.load(r"your_directory\n2u_chem_common_230724.npy", allow_pickle=True); print(n2u_chem_mat.shape)
male_chem_mat = np.load(r"your_directory\male_chem_common_230724.npy", allow_pickle=True); print(male_chem_mat.shape)
dauer_chem_mat = np.load(r"your_directory\dauer_chem_common_230724.npy", allow_pickle=True); print(dauer_chem_mat.shape)

n2u_gap_mat = np.load(r"your_directory\n2u_gap_ori_180_230724.npy", allow_pickle=True); print(n2u_gap_mat.shape)
male_gap_mat = np.load(r"your_directory\male_gap_common_230724.npy", allow_pickle=True); print(male_gap_mat.shape)
dauer_gap_mat = np.load(r"your_directory\dauer1_gap_ori_180_230724.npy", allow_pickle=True); print(dauer_gap_mat.shape)


# Main

In [ ]:
# extract submatrix and apply 'Methods' from Pechuck 2022

# submatrix
noci_idx = [32,33, 42,43, 44,45, 46,47, 100,101]  # 230724 ver.
    # target neurons; ASH, AVA, AVB, AVD, PVC (LR pair)

n2u_noci_chem = np.copy(n2u_chem_mat[np.ix_(noci_idx,noci_idx)])
n2u_noci_gap = np.copy(n2u_gap_mat[np.ix_(noci_idx,noci_idx)])
male_noci_chem = np.copy(male_chem_mat[np.ix_(noci_idx,noci_idx)])
male_noci_gap = np.copy(male_gap_mat[np.ix_(noci_idx,noci_idx)])
dauer_noci_chem = np.copy(dauer_chem_mat[np.ix_(noci_idx,noci_idx)])
dauer_noci_gap = np.copy(dauer_gap_mat[np.ix_(noci_idx,noci_idx)])

# LR pair merge
n2u_noci_chem_merge = np.zeros((5,5)); n2u_noci_gap_merge = np.zeros((5,5))  # change size
male_noci_chem_merge = np.zeros((5,5)); male_noci_gap_merge = np.zeros((5,5))  # change size
dauer_noci_chem_merge = np.zeros((5,5)); dauer_noci_gap_merge = np.zeros((5,5))  # change size


In [ ]:
# chemical
for ii in range(n2u_noci_chem_merge.shape[0]):
    for jj in range(n2u_noci_chem_merge.shape[1]):
        square = n2u_noci_chem[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2(or 4?) vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            n2u_noci_chem_merge[ii,jj] = square_avg

for ii in range(male_noci_chem_merge.shape[0]):
    for jj in range(male_noci_chem_merge.shape[1]):
        square = male_noci_chem[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            male_noci_chem_merge[ii,jj] = square_avg

for ii in range(dauer_noci_chem_merge.shape[0]):
    for jj in range(dauer_noci_chem_merge.shape[1]):
        square = dauer_noci_chem[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            dauer_noci_chem_merge[ii,jj] = square_avg


# gap junction
for ii in range(n2u_noci_gap_merge.shape[0]):
    for jj in range(n2u_noci_gap_merge.shape[1]):
        square = n2u_noci_gap[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            n2u_noci_gap_merge[ii,jj] = square_avg

for ii in range(male_noci_gap_merge.shape[0]):
    for jj in range(male_noci_gap_merge.shape[1]):
        square = male_noci_gap[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            male_noci_gap_merge[ii,jj] = square_avg

for ii in range(dauer_noci_gap_merge.shape[0]):
    for jj in range(dauer_noci_gap_merge.shape[1]):
        square = dauer_noci_gap[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            dauer_noci_gap_merge[ii,jj] = square_avg


In [ ]:
# check full
print_df = pd.DataFrame(data=dauer_noci_chem,\
                        index=[com_id[noci_idx,0]], columns=[com_id[noci_idx,0]])
print_df

In [ ]:
# check merged
print_df = pd.DataFrame(data=dauer_noci_chem_merge,\
                        index=["ASH","AVA","AVB","AVD","PVC"],\
                        columns=["ASH","AVA","AVB","AVD","PVC"])
print_df

---

### Normalize according to 'norm180_comp' ver.

In [ ]:
# from normalized matrix
n2u_noci_chem_norm = np.copy(norm180_comp[:,:,9][np.ix_(noci_idx,noci_idx)])
dauer_noci_chem_norm = np.copy(norm180_comp[:,:,10][np.ix_(noci_idx,noci_idx)])

# LR pair merge
n2u_noci_chem_merge_norm = np.zeros((5,5))
dauer_noci_chem_merge_norm = np.zeros((5,5))

# chemical
for ii in range(n2u_noci_chem_merge_norm.shape[0]):
    for jj in range(n2u_noci_chem_merge_norm.shape[1]):
        square = n2u_noci_chem_norm[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
        if ii==jj:
            continue
        else:
            n2u_noci_chem_merge_norm[ii,jj] = square_avg

for ii in range(dauer_noci_chem_merge_norm.shape[0]):
    for jj in range(dauer_noci_chem_merge_norm.shape[1]):
        square = dauer_noci_chem_norm[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
        if ii==jj:
            continue
        else:
            dauer_noci_chem_merge_norm[ii,jj] = square_avg


In [ ]:
# reshape dauer nociception circuit into Pechuk order

new_order = [0,1,3,2,4]

# adjust strength
mult_norm = np.ones((5,5))  # factor to multiply to dauer normalized matrix

ori_mat_temp = np.copy(n2u_noci_chem_merge[np.ix_(new_order,new_order)])
norm_mat_temp = np.copy(n2u_noci_chem_merge_norm[np.ix_(new_order,new_order)])
for ii in range(mult_norm.shape[0]):
    for jj in range(mult_norm.shape[1]):
        if norm_mat_temp[ii,jj]==0:
            continue
        else:
            mult_norm[ii,jj] = ori_mat_temp[ii,jj] / norm_mat_temp[ii,jj]

print(mult_norm)  # find the most frequent value (except 1)


In [ ]:
# check dataframe
print("  N2U (Cook)")
print_df = pd.DataFrame(data=np.round(ori_mat_temp,3),\
                        index=["ASH","AVA","AVB","AVD","PVC"],\
                        columns=["ASH","AVA","AVB","AVD","PVC"])
print_df

In [ ]:
# check dataframe
print("  N2U (norm.)")
print_df = pd.DataFrame(data=np.round(norm_mat_temp,3),\
                        index=["ASH","AVA","AVB","AVD","PVC"],\
                        columns=["ASH","AVA","AVB","AVD","PVC"])
print_df

In [ ]:
# check dataframe
print("  N2U (scaled)")
print_df = pd.DataFrame(data=np.round(norm_mat_temp * mult_norm[0,1],3),\
                        index=["ASH","AVA","AVB","AVD","PVC"],\
                        columns=["ASH","AVA","AVB","AVD","PVC"])
print_df

---

In [ ]:
import extract_connectivity_df as ecd
import supporting_functions as sf

path_to_cook_file = r"your_directory\SI_5_cook_original.xlsx"

herm_chem_full_connectivity = pd.read_excel(path_to_cook_file, sheet_name = "hermaphrodite chemical", header = None)
herm_gapj_full_connectivity = pd.read_excel(path_to_cook_file, sheet_name = "hermaphrodite gap jn symmetric", header = None)
male_chem_full_connectivity = pd.read_excel(path_to_cook_file, sheet_name = "male chemical", header = None)
male_gapj_full_connectivity = pd.read_excel(path_to_cook_file, sheet_name = "male gap jn symmetric", header = None)

# =============================================================================
# Define neurons of interest:
# In circuit_neurons the keys are the name of the neural class, and the values are
# the regex expressions to find the cells in Cook's data.  
# =============================================================================
circuit_neurons = {'ASH':'ASH[RL]', 'AVA':'AVA[RL]', 'AVD':'AVD[RL]',\
                   'AVB':'AVB[RL]', 'PVC':'PVC[RL]', 'A':'[DV]A[0-9]', 'B':'[DV]B[0-9]'}
neurons = list(circuit_neurons.keys())

Sgap_herm_head,Schem_herm_head,Sgap_male_head,Schem_male_head = ecd.final_connectivity_matrix(herm_chem_full_connectivity,\
                                                                                              herm_gapj_full_connectivity,\
                                                                                              male_chem_full_connectivity,\
                                                                                              male_gapj_full_connectivity,\
                                                                                              circuit_neurons)


In [ ]:
# reshape dauer nociception circuit into Pechuk order
    # Original order = ASH, AVA, AVB, AVD, PVC
    #   Pechuk order = ASH, AVA, AVD, AVB, PVC
    # assume that connectivity to A and B is similar to hermaphrodite (i.e., use hermaphrodite's data)

Schem_dauer_head = np.zeros((7,7))  # mind the direction (transpose)
Sgap_dauer_head = np.zeros((7,7))  # mind the direction (transpose)

# chemical
dauer_chem_adjusted = np.copy( (dauer_noci_chem_merge_norm[np.ix_(new_order,new_order)] * mult_norm[0,1]) )  # Pechuk order
Schem_dauer_head[:5,:5] = np.copy(dauer_chem_adjusted.T)  # transpose
Schem_dauer_head[5:,:] = np.copy(Schem_herm_head[5:,:])  # A, B row (from herm.)
Schem_dauer_head[:,5:] = np.copy(Schem_herm_head[:,5:])  # A, B column (from herm.)

# electrical (don't need to adjust strength; strength by section number)
dauer_gap_adjusted = np.copy( dauer_noci_gap_merge[np.ix_(new_order,new_order)] )  # Pechuk order
Sgap_dauer_head[:5,:5] = np.copy(dauer_gap_adjusted.T)
Sgap_dauer_head[5:,:] = np.copy(Sgap_herm_head[5:,:])
Sgap_dauer_head[:,5:] = np.copy(Sgap_herm_head[:,5:])


In [ ]:
# check
print("Dauer")
print_df_1 = pd.DataFrame(data=np.round(Sgap_dauer_head[:5,:5].T,2),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])  # Schem vs. Sgap
print_df_1

In [ ]:
# check
print("Herm.")
print_df_2 = pd.DataFrame(data=np.round(Sgap_herm_head[:5,:5].T,2),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])  # Schem vs. Sgap
print_df_2

### Identify dimorphic connections

In [ ]:
# identify dimorphic connections

# chemical
Schem_herm_head_bi = np.copy(Schem_herm_head); Schem_herm_head_bi[Schem_herm_head_bi!=0] = 1
Schem_dauer_head_bi = np.copy(Schem_dauer_head); Schem_dauer_head_bi[Schem_dauer_head_bi!=0] = 1

chem_diff_binary = Schem_herm_head_bi - Schem_dauer_head_bi  # herm-specific = 1, dauer-specific = -1

chem_diff_fold = np.zeros(chem_diff_binary.shape)
for ii in range(chem_diff_fold.shape[0]):
    for jj in range(chem_diff_fold.shape[1]):
        v1 = Schem_herm_head[ii,jj]
        v2 = Schem_dauer_head[ii,jj]
        if v1==0 or v2==0:
            continue
        else:
            if max(v1,v2) / min(v1,v2) > 1.5:  # change thresh
                chem_diff_fold[ii,jj] = 1
            else:
                continue

print("Chem, Bi:", np.nonzero(chem_diff_binary)[0].size, "\t", "Chem, Fold:", np.nonzero(chem_diff_fold)[0].size)

# gap
Sgap_herm_head_bi = np.copy(Sgap_herm_head); Sgap_herm_head_bi[Sgap_herm_head_bi!=0] = 1
Sgap_dauer_head_bi = np.copy(Sgap_dauer_head); Sgap_dauer_head_bi[Sgap_dauer_head_bi!=0] = 1

gap_diff_binary = Sgap_herm_head_bi - Sgap_dauer_head_bi  # herm-specific = 1, dauer-specific = -1

gap_diff_fold = np.zeros(gap_diff_binary.shape)
for ii in range(gap_diff_fold.shape[0]):
    for jj in range(gap_diff_fold.shape[1]):
        v1 = Sgap_herm_head[ii,jj]
        v2 = Sgap_dauer_head[ii,jj]
        if v1==0 or v2==0:
            continue
        else:
            if max(v1,v2) / min(v1,v2) > 1.5:  # change thresh
                gap_diff_fold[ii,jj] = 1
            else:
                continue

print(" Gap, Bi:", int(np.nonzero(gap_diff_binary)[0].size/2), "\t", " Gap, Fold:", int(np.nonzero(gap_diff_fold)[0].size/2))


In [ ]:
# print neuron

## ============================================================
# NOTE :: matrix is transposed (column = pre; row = post)
## ============================================================

n_name = ["ASH", "AVA", "AVD", "AVB", "PVC", "A", "B"]

print("Chem, Binary :")
for ii in range(chem_diff_binary.shape[0]):
    for jj in range(chem_diff_binary.shape[1]):
        if chem_diff_binary[ii,jj]!=0:
            print(" ",n_name[jj],">",n_name[ii])
        else:
            continue

print("\nChem, Fold :")
for ii in range(chem_diff_fold.shape[0]):
    for jj in range(chem_diff_fold.shape[1]):
        if chem_diff_fold[ii,jj]!=0:
            print(" ",n_name[jj],">",n_name[ii])
        else:
            continue

print("\nGap, Binary :")
for ii in range(gap_diff_binary.shape[0]):
    for jj in range(gap_diff_binary.shape[1]):
        if ii<jj: 
            if gap_diff_binary[ii,jj]!=0:
                print(" ",n_name[jj],"-",n_name[ii])
            else:
                continue
        else:
            continue

print("\nGap, Fold :")
for ii in range(gap_diff_fold.shape[0]):
    for jj in range(gap_diff_fold.shape[1]):
        if ii<jj: 
            if gap_diff_fold[ii,jj]!=0:
                print(" ",n_name[jj],"-",n_name[ii])
            else:
                continue
        else:
            continue


In [ ]:
# non-zero indices =================================================

cb_nz = np.nonzero(chem_diff_binary) # chem, bi
cf_nz = np.nonzero(chem_diff_fold)   # chem, fold
gb_nz = np.nonzero(gap_diff_binary) # gap, bi
gf_nz = np.nonzero(gap_diff_fold)   # gap, fold

# ==================================================================

In [ ]:
# Note that nonzero indices for gap junction matrix contain 'duplicates'

print(gf_nz[0])
print(gf_nz[1])


In [ ]:
# check index of connection to flip
    # ex) ASH - AVD gap (gap, binary, index 0)

POST = gb_nz[0]  # change (Note that matrix is transposed)
PRE = gb_nz[1]   # change (column=pre; row=post)

IDX = 4  # change

print(n_name[PRE[IDX]], " ", n_name[POST[IDX]])


---

In [ ]:
# (temp) identify dimorphic connections - strength

# chemical
Schem_herm_head_bi = np.copy(Schem_herm_head); Schem_herm_head_bi[Schem_herm_head_bi!=0] = 1
Schem_dauer_head_bi = np.copy(Schem_dauer_head); Schem_dauer_head_bi[Schem_dauer_head_bi!=0] = 1

chem_diff_binary_val = Schem_herm_head_bi - Schem_dauer_head_bi  # herm-specific = 1, dauer-specific = -1

chem_diff_fold_val = np.zeros(chem_diff_binary.shape)
for ii in range(chem_diff_fold_val.shape[0]):
    for jj in range(chem_diff_fold_val.shape[1]):
        v1 = Schem_herm_head[ii,jj]
        v2 = Schem_dauer_head[ii,jj]
        if v1==0 or v2==0:
            continue
        else:
            if max(v1,v2) / min(v1,v2) > 1.5:  # change thresh
                chem_diff_fold_val[ii,jj] = ( max(v1,v2) / min(v1,v2) )
            else:
                continue

print("Chem, Bi:", np.nonzero(chem_diff_binary_val)[0].size, "\t", "Chem, Fold:", np.nonzero(chem_diff_fold_val)[0].size)

# gap
Sgap_herm_head_bi = np.copy(Sgap_herm_head); Sgap_herm_head_bi[Sgap_herm_head_bi!=0] = 1
Sgap_dauer_head_bi = np.copy(Sgap_dauer_head); Sgap_dauer_head_bi[Sgap_dauer_head_bi!=0] = 1

gap_diff_binary_val = Sgap_herm_head_bi - Sgap_dauer_head_bi  # herm-specific = 1, dauer-specific = -1

gap_diff_fold_val = np.zeros(gap_diff_binary.shape)
for ii in range(gap_diff_fold_val.shape[0]):
    for jj in range(gap_diff_fold_val.shape[1]):
        v1 = Sgap_herm_head[ii,jj]
        v2 = Sgap_dauer_head[ii,jj]
        if v1==0 or v2==0:
            continue
        else:
            if max(v1,v2) / min(v1,v2) > 1.5:  # change thresh
                gap_diff_fold_val[ii,jj] = ( max(v1,v2) / min(v1,v2) )
            else:
                continue

print(" Gap, Bi:", int(np.nonzero(gap_diff_binary_val)[0].size/2), "\t", " Gap, Fold:", int(np.nonzero(gap_diff_fold_val)[0].size/2))


---

## FLIP

In [ ]:
print(cf_nz[0])
print(cf_nz[1])
print(cb_nz[0])
print(cb_nz[1])

In [ ]:
# make 'flipped' matrices [CHEM]

Schem_herm_head_temp = np.copy(Schem_herm_head)
Schem_dauer_head_temp = np.copy(Schem_dauer_head)

POST = cf_nz[0]  # change (Note that matrix is transposed)
PRE = cf_nz[1]   # change (column=pre; row=post)

IDX = 4

Schem_dauer_head_temp[POST[IDX],PRE[IDX]] = Schem_herm_head[POST[IDX],PRE[IDX]]  # flip

Schem_herm_head_flip = np.copy(Schem_herm_head_temp)
Schem_dauer_head_flip = np.copy(Schem_dauer_head_temp)


In [ ]:
# make 'deleted' matrices [CHEM]

Schem_herm_head_temp = np.copy(Schem_herm_head)
Schem_dauer_head_temp = np.copy(Schem_dauer_head)

POST = cb_nz[0]  # change (Note that matrix is transposed)
PRE = cb_nz[1]   # change (column=pre; row=post)

IDX = 1

Schem_herm_head_temp[POST[IDX],PRE[IDX]] = 0  # delete
Schem_dauer_head_temp[POST[IDX],PRE[IDX]] = 0  # delete

Schem_herm_head_flip = np.copy(Schem_herm_head_temp)
Schem_dauer_head_flip = np.copy(Schem_dauer_head_temp)


In [ ]:
# check flip
print("original")
print_df = pd.DataFrame(data=np.round(Schem_herm_head[:5,:5],3),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])
print_df

In [ ]:
# check flip
print("flipped")
print_df = pd.DataFrame(data=np.round(Schem_herm_head_flip[:5,:5],3),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])
print_df

---
---

In [ ]:
# Note that indices appear twice for the same connection

print(gf_nz[0])
print(gf_nz[1])
print(gb_nz[0])
print(gb_nz[1])


- Fold index : 0, 2, 3, 4, 6 <br><br>
- Binary index : 0, 2, 4

In [ ]:
# make 'flipped' matrices [GAP]

Sgap_herm_head_temp = np.copy(Sgap_herm_head)
Sgap_dauer_head_temp = np.copy(Sgap_dauer_head)

POST = gf_nz[0]  # change (Note that matrix is transposed)
PRE = gf_nz[1]   # change (column=pre; row=post)

IDX = 3  # change

Sgap_herm_head_temp[POST[IDX],PRE[IDX]] = Sgap_dauer_head[POST[IDX],PRE[IDX]]  # flip 1

Sgap_herm_head_temp[PRE[IDX],POST[IDX]] = Sgap_dauer_head[PRE[IDX],POST[IDX]]  # flip 2

Sgap_herm_head_flip = np.copy(Sgap_herm_head_temp)
Sgap_dauer_head_flip = np.copy(Sgap_dauer_head_temp)


In [ ]:
# make 'deleted' matrices [GAP] - mostly for dauer network

Sgap_herm_head_temp = np.copy(Sgap_herm_head)
Sgap_dauer_head_temp = np.copy(Sgap_dauer_head)


POST = gf_nz[0]  # change (Note that matrix is transposed)
PRE = gf_nz[1]   # change (column=pre; row=post)

IDX = 0  # change

Sgap_dauer_head_temp[POST[IDX],PRE[IDX]] = 0  # delete 1
Sgap_dauer_head_temp[PRE[IDX],POST[IDX]] = 0  # delete 2


POST = gf_nz[0]  # change (Note that matrix is transposed)
PRE = gf_nz[1]   # change (column=pre; row=post)

IDX = 3  # change

Sgap_dauer_head_temp[POST[IDX],PRE[IDX]] = 0  # delete 1
Sgap_dauer_head_temp[PRE[IDX],POST[IDX]] = 0  # delete 2


# Sgap_herm_head_temp[0,1] = 0  # manual deletion
# Sgap_herm_head_temp[1,0] = 0  # manual deletion


Sgap_herm_head_flip = np.copy(Sgap_herm_head_temp)
Sgap_dauer_head_flip = np.copy(Sgap_dauer_head_temp)


In [ ]:
# (temp) make 'deleted' matrices [CHEM + GAP]
    # remove ASH == AVA gap
    # remove ASH -> AVA chem
    # in dauer

Schem_herm_head_temp = np.copy(Schem_herm_head)
Schem_dauer_head_temp = np.copy(Schem_dauer_head)
Sgap_herm_head_temp = np.copy(Sgap_herm_head)
Sgap_dauer_head_temp = np.copy(Sgap_dauer_head)


Schem_dauer_head_temp[1,0] = 0  # chem delete
    # ASH -> AVA removed
Schem_dauer_head_temp[3,0] = 0  # chem delete
    # ASH -> AVB removed


POST = gf_nz[0]  # change (Note that matrix is transposed)
PRE = gf_nz[1]   # change (column=pre; row=post)

IDX = 0  # change

Sgap_dauer_head_temp[POST[IDX],PRE[IDX]] = 0  # delete 1
Sgap_dauer_head_temp[PRE[IDX],POST[IDX]] = 0  # delete 2
    # ASH == AVA removed

Schem_herm_head_flip = np.copy(Schem_herm_head_temp)
Schem_dauer_head_flip = np.copy(Schem_dauer_head_temp)
Sgap_herm_head_flip = np.copy(Sgap_herm_head_temp)
Sgap_dauer_head_flip = np.copy(Sgap_dauer_head_temp)


In [ ]:
# (temp) make 'deleted' matrices [CHEM + GAP]
    # remove ASH == AVA gap
    # remove ASH -> AVA chem
    # in adult

Schem_herm_head_temp = np.copy(Schem_herm_head)
Schem_dauer_head_temp = np.copy(Schem_dauer_head)
Sgap_herm_head_temp = np.copy(Sgap_herm_head)
Sgap_dauer_head_temp = np.copy(Sgap_dauer_head)


Schem_herm_head_temp[1,0] = 0  # chem delete
    # ASH -> AVA removed
Schem_herm_head_temp[3,0] = 0  # chem delete
    # ASH -> AVB removed

Sgap_herm_head_temp[1,0] = 0  # delete 1
Sgap_herm_head_temp[0,1] = 0  # delete 2
    # ASH == AVA removed

Schem_herm_head_temp[2,0] = 9.5  # masculinization


Schem_herm_head_flip = np.copy(Schem_herm_head_temp)
Schem_dauer_head_flip = np.copy(Schem_dauer_head_temp)
Sgap_herm_head_flip = np.copy(Sgap_herm_head_temp)
Sgap_dauer_head_flip = np.copy(Sgap_dauer_head_temp)


---

In [ ]:
# check flip
print("original")
print_df = pd.DataFrame(data=np.round(Schem_herm_head[:5,:5],3),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])
print_df

In [ ]:
# check flip
print("flipped")
print_df = pd.DataFrame(data=np.round(Schem_herm_head_flip[:5,:5],3),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])
print_df

# Simulation application (with `flipped`/`deleted` matrices)
> original code from *Pechuk et al 2022*

### Use `flipped`/`deleted` matrix:
> check which connection is flipped/deleted <br><br>
> (check whether `Chem` or `Gap`)

In [ ]:
# # =============================================================================
# mutual_sets = pd.read_csv(r"your_directory\herm_cell_activity_basic_mutual_herm_male_dauer_667.csv")  # 187 sets

# Parameters_df = pd.read_csv(r"your_directory\Parameters_combinations_df.csv", index_col = 0)  # original combination from Pechuk

# pol_fname = r"your_directory\Polarity_combinations_sample.pkl"  # original combination from Pechuk
# possible_combinations_df = pd.read_pickle(pol_fname)


# # just copy colums 'index', 'parameters_index', and 'polarity_index'
#     # run again with this dataframe
# df_temp = pd.DataFrame()
# df_temp.loc[:, 'index'] = mutual_sets.loc[:, 'index']
# df_temp.loc[:, 'parameters_index'] = mutual_sets.loc[:, 'parameters_index']
# df_temp.loc[:, 'polarity_index'] = mutual_sets.loc[:, 'polarity_index']
# print(df_temp.head(5))
# # =============================================================================

In [ ]:
# df_temp.to_csv(r"your_directory\valid_188_initial_df.csv")

---

In [ ]:
# =============================================================================
# Set environment
# =============================================================================
# define neurons of interest:
circuit_neurons = {'ASH':'ASH[RL]', 'AVA':'AVA[RL]', 'AVD':'AVD[RL]', 'AVB':'AVB[RL]', 'PVC':'PVC[RL]', 'A':'[DV]A[0-9]', 'B':'[DV]B[0-9]'}
neurons = list(circuit_neurons.keys())

# time:
t = np.linspace(0,15,75001) # 15 seconds, in 0.0002 (0.2ms) jumps
correction_to_ms = int(round(0.001/(t[1]-t[0])))
sensory_start = 5000 * correction_to_ms
sensory_end = 10000 * correction_to_ms

# resting state: 
Vrest = np.reshape(np.repeat(np.array([-0.04]),len(neurons)), (len(neurons),1)) #units: V

# =============================================================================
# Input that would trigger a specific voltage change:
# =============================================================================
# V = IR, I = V/R
sensory_input_calc = lambda res, volt: volt/res
sensory_voltage_changes = [75 * 10 **-3, 65 * 10 **-3, 55 * 10 ** -3, 45 * 10 **-3, 35 * 10 **-3, 25 * 10 ** -3, 15 * 10 **-3, 5 * 10 ** -3, 1 * 10 **-3]


In [ ]:
# running 667 sets on flipped network
    # [IMPORTANT] check FLIPPED matrices

# =============================================================================
# =============================================================================
Parameters_df = pd.read_csv(r"your_directory\Parameters_combinations_df.csv", index_col = 0)  # original combination from Pechuk

pol_fname = r"your_directory\Polarity_combinations_sample.pkl"  # original combination from Pechuk
possible_combinations_df = pd.read_pickle(pol_fname)

df_temp = pd.read_csv(r"your_directory\valid_667_initial_df.csv", index_col=0)  # 667 sets initial condition

# =============================================================================
# =============================================================================


Parameters_Job = df_temp.copy(); print(Parameters_Job.shape)

# =============================================================================
# initializing the conditions
# =============================================================================
herm_basic_conditions = []; herm_cell_act_conditions = []
dauer_basic_conditions = []; dauer_cell_act_conditions = []
Sgap_herm_adj = []; Schem_herm_adj = []
Sgap_dauer_adj = []; Schem_dauer_adj = []
time_constant = []

row_cnt = 0; iter_cnt = 1
CUT = 1* 10**1
iter_num = int(Parameters_Job.shape[0] / CUT) + 1

rand_row = np.array(Parameters_Job.index)
for row in rand_row:
    parameter_ind = Parameters_Job.parameters_index[row]
    polarity_ind = Parameters_Job.polarity_index[row]
# =============================================================================
# Randomly adjusting the strength of the connections
# =============================================================================
    Schem_herm_row, mult_chem_herm = ecd.adjusting_strengths(Schem_herm_head_flip,[1.2, 1, 0.8], "chem")  # [ check _FLIP ]
    Sgap_herm_row, mult_gap_herm =  ecd.adjusting_strengths(Sgap_herm_head_flip,[1.2, 1, 0.8], "gap")  # [ check _FLIP ]
    Schem_dauer_row, mult_chem_dauer =  ecd.adjusting_strengths(Schem_dauer_head_flip,[1.2, 1, 0.8], "chem")  # [ check _FLIP ]
    Sgap_dauer_row, mult_gap_dauer =  ecd.adjusting_strengths(Sgap_dauer_head_flip,[1.2, 1, 0.8], "gap")  # [ check _FLIP ]

# =============================================================================
# Update polarity
# =============================================================================
    Schem_herm_row = sf.adjust_polarity(Schem_herm_row, neurons, polarity_ind, possible_combinations_df)
    Schem_dauer_row = sf.adjust_polarity(Schem_dauer_row, neurons, polarity_ind, possible_combinations_df)

    Sgap_herm_adj.append(Sgap_herm_row)
    Schem_herm_adj.append(Schem_herm_row)
    Sgap_dauer_adj.append(Sgap_dauer_row)
    Schem_dauer_adj.append(Schem_dauer_row)

# =============================================================================
# Define input current:
# =============================================================================
    I = sf.input_current(Sgap_herm_row, t, range(sensory_start, sensory_end), neurons.index("ASH"),\
                         Parameters_df.Sensory_Input[parameter_ind], 0, range(0,len(t)), neurons.index("AVA"),\
                         Parameters_df.I_AVA[parameter_ind], 0, range(0,len(t)), neurons.index("AVB"),\
                         Parameters_df.I_AVB[parameter_ind], 0, range(0,len(t)),  neurons.index("PVC"),\
                         Parameters_df.I_PVC[parameter_ind], 0) # units: A

# =============================================================================
# Hermaphrodites
# =============================================================================
    sex = "Hermaphrodites"
    V_h_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],\
                                     Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind],\
                                     Parameters_df.Gap_Conductance[parameter_ind], I, Schem_herm_row, Sgap_herm_row, sex)
    basic_conditions_herm = sf.basic_conditions_check(V_h_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)
    herm_basic_conditions.append(basic_conditions_herm)
    if basic_conditions_herm == (True):
        herm_cell_act_conditions.append(sf.cells_activity_conditions_check(V_h_adj_con, neurons, sensory_start, sensory_end, correction_to_ms))
    else:
        herm_cell_act_conditions.append(("irrelevant"))
# =============================================================================
# Dauers
# =============================================================================
    sex = "Hermaphrodites"  # this is correct
    V_d_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],\
                                     Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind],\
                                     Parameters_df.Gap_Conductance[parameter_ind], I, Schem_dauer_row, Sgap_dauer_row, sex)  # changed Schem/Sgap matrices
    basic_conditions_dauer = sf.basic_conditions_check(V_d_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)
    dauer_basic_conditions.append(basic_conditions_dauer)
    if basic_conditions_dauer == (True):
        dauer_cell_act_conditions.append(sf.cells_activity_conditions_check(V_d_adj_con, neurons,sensory_start, sensory_end, correction_to_ms))
    else:
        dauer_cell_act_conditions.append(("irrelevant"))

# =============================================================================
# General
# =============================================================================
    time_constant.append(Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind] < 500 * 10 **-3)
    row_cnt += 1
    
    if row_cnt % CUT == 0:
        print("_ _ _ finished row", row_cnt, "_ _ _", iter_cnt, "/", iter_num)
        iter_cnt += 1
    else:
        continue

# =============================================================================
# Save to dataframe
# =============================================================================
Parameters_Job["herm_basic_conditions"] = herm_basic_conditions
Parameters_Job["dauer_basic_conditions"] = dauer_basic_conditions
Parameters_Job["herm_cell_act_conditions"] = herm_cell_act_conditions
Parameters_Job["dauer_cell_act_conditions"] = dauer_cell_act_conditions
Parameters_Job["time_constant"] = time_constant
Parameters_Job["Schem_herm_adj"] = Schem_herm_adj
Parameters_Job["Sgap_herm_adj"] = Sgap_herm_adj
Parameters_Job["Schem_dauer_adj"] = Schem_dauer_adj
Parameters_Job["Sgap_dauer_adj"] = Sgap_dauer_adj
Parameters_Job["True_herm"] = ((Parameters_Job["herm_basic_conditions"] == True) | (Parameters_Job["herm_basic_conditions"] == "True")) & ((Parameters_Job["herm_cell_act_conditions"] == True) | (Parameters_Job["herm_cell_act_conditions"] == "True"))
Parameters_Job["True_dauer"] = ((Parameters_Job["dauer_basic_conditions"] == True) | (Parameters_Job["dauer_basic_conditions"] == "True")) & ((Parameters_Job["dauer_cell_act_conditions"] == True) | (Parameters_Job["dauer_cell_act_conditions"] == "True"))  
Parameters_Job["mutual_basic_herm_activity"] = ((Parameters_Job["herm_basic_conditions"] == True) | (Parameters_Job["herm_basic_conditions"] == "True")) & ((Parameters_Job["herm_cell_act_conditions"] == True) | (Parameters_Job["herm_cell_act_conditions"] == "True")) & ((Parameters_Job["dauer_basic_conditions"] == True) | (Parameters_Job["dauer_basic_conditions"] == "True"))


In [ ]:
# save to csv

Parameters_Job.to_csv(r"your_directory\parameters_basic_conditions_del_gc_ASH-AVA_ASH-AVB_adult.csv")  # [ CHANGE NAME ]


In [ ]:
# select mutual parameter sets (flip ver.)

from os import path
import random
import supporting_functions as sf

# =============================================================================
single_run_df = pd.read_csv(r"your_directory\parameters_basic_conditions_del_gc_ASH-AVA_ASH-AVB_adult.csv", index_col=0)  # [ CHANGE ]
single_run_df["parameters_index"] = single_run_df["parameters_index"].astype("int32")
single_run_df["polarity_index"] = single_run_df["polarity_index"].astype("int8")
some_true =  single_run_df[((single_run_df["herm_basic_conditions"] == "True") | (single_run_df["herm_basic_conditions"] == True) | (single_run_df["dauer_basic_conditions"] == "True") | (single_run_df["dauer_basic_conditions"] == True)) & (single_run_df["time_constant"] == True)] 

parameters_pol_samp_df = some_true.copy()
print(parameters_pol_samp_df.info())
parameters_pol_samp_df.to_csv(r"your_directory\basic_true_par_pol_samp_del_gc_ASH-AVA_ASH-AVB_adult.csv")  # [ CHANGE ]
herm_cell_activity_basic_mutual = parameters_pol_samp_df[(parameters_pol_samp_df["mutual_basic_herm_activity"] == "True") | (parameters_pol_samp_df["mutual_basic_herm_activity"] == True)].reset_index()
herm_cell_activity_basic_mutual.to_csv(r"your_directory\herm_cell_activity_basic_mutual_del_gc_ASH-AVA_ASH-AVB_adult.csv")  # [ CHANGE ]
# =============================================================================


In [ ]:
# # =============================================================================
# # Define for cluster use
# # =============================================================================
# job_ID = int(os.environ['LSB_JOBINDEX']) - 1
job_num = 2000

# # =============================================================================
# # Read parameters sets
# # this file contains all sets that were appropriate to either of the sexes, 
# # after the adjustments to the connections' strength. 
# # =============================================================================
fname1 = r"your_directory\herm_cell_activity_basic_mutual_del_gc_ASH-AVA_ASH-AVB_adult.csv"  # [ CHANGE ]
herm_cell_activity_basic_mutual_flip = pd.read_csv(fname1)  # check variable name

fname2 = r"your_directory\Parameters_combinations_df.csv"  # original
Parameters_df = pd.read_csv(fname2, index_col = 0)


In [ ]:
# =============================================================================
# Set environment
# =============================================================================
# define neurons of interest:
circuit_neurons = {'ASH':'ASH[RL]', 'AVA':'AVA[RL]', 'AVD':'AVD[RL]', 'AVB':'AVB[RL]', 'PVC':'PVC[RL]', 'A':'[DV]A[0-9]', 'B':'[DV]B[0-9]'}
neurons = list(circuit_neurons.keys())

# time:
t = np.linspace(0,15,75001) # 15 seconds, in 0.0002 (0.2ms) jumps
correction_to_ms = int(round(0.001/(t[1]-t[0])))
sensory_start = 5000 * correction_to_ms
sensory_end = 10000 * correction_to_ms

# resting state: 
Vrest = np.reshape(np.repeat(np.array([-0.04]),len(neurons)), (len(neurons),1)) #units: V

# =============================================================================
# Input that would trigger a specific voltage change:
# =============================================================================
# V = IR, I = V/R
sensory_input_calc = lambda res, volt: volt/res
sensory_voltage_changes = [75 * 10 **-3, 65 * 10 **-3, 55 * 10 ** -3, 45 * 10 **-3, 35 * 10 **-3, 25 * 10 ** -3, 15 * 10 **-3, 5 * 10 ** -3, 1 * 10 **-3]


Different_stimulus_Job = herm_cell_activity_basic_mutual_flip.copy()  # check name


# =============================================================================
# Initializing response arrays
# =============================================================================
herm_response = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
dauer_response = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan

# =============================================================================
# Simulating sensory stimuli in varying strength
# =============================================================================
row_cnt = 0
for row in range(Different_stimulus_Job.index[0],Different_stimulus_Job.index[len(Different_stimulus_Job)-1]+1):
    parameter_ind = Different_stimulus_Job.parameters_index[row] 
    Sgap_herm = sf.str_to_nparray(Different_stimulus_Job.Sgap_herm_adj[row], (len(neurons), len(neurons)))  # 'flip' already applied
    Schem_herm = sf.str_to_nparray(Different_stimulus_Job.Schem_herm_adj[row],  (len(neurons), len(neurons)))  # 'flip' already applied
    Sgap_dauer = sf.str_to_nparray(Different_stimulus_Job.Sgap_dauer_adj[row],  (len(neurons), len(neurons)))  # 'flip' already applied
    Schem_dauer = sf.str_to_nparray(Different_stimulus_Job.Schem_dauer_adj[row],  (len(neurons), len(neurons)))  # 'flip' already applied
    for voltage in range(len(sensory_voltage_changes)):
        sensory_input = sensory_input_calc(Parameters_df.Rin[parameter_ind], sensory_voltage_changes[voltage])
        I = sf.input_current(Sgap_herm, t, range(sensory_start, sensory_end), neurons.index("ASH"), sensory_input,\
                             0, range(0,len(t)), neurons.index("AVA"), Parameters_df.I_AVA[parameter_ind],\
                             0, range(0,len(t)), neurons.index("AVB"), Parameters_df.I_AVB[parameter_ind],\
                             0, range(0,len(t)), neurons.index("PVC"), Parameters_df.I_PVC[parameter_ind], 0) # units: A
        
# =============================================================================
# Hermaphrodites
# =============================================================================
        V_h = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_herm, Sgap_herm)
# since we expect oscillations we check the average of 1 second to determine which is higher, and not a single time step. 
        herm_response[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("A"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("B"), sensory_end - (1000 * correction_to_ms) : sensory_end])

# =============================================================================
# Dauers
# =============================================================================
        V_d = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_dauer, Sgap_dauer)
# since we expect oscillations we check the average of 1 second to determine which is higher, and not a single time step. 
        dauer_response[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("A"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("B"), sensory_end - (1000 * correction_to_ms) : sensory_end])
    
    row_cnt += 1
    
    if row_cnt % 10**1 == 0:
        print("finished row "+str(row_cnt))
    else:
        continue

print(row_cnt, "Done")


for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["herm_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["dauer_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["herm_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(herm_response[:,sens_input]> 0, 1, 0)

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(dauer_response[:,sens_input]> 0, 1, 0)


In [ ]:
Different_stimulus_Job.to_csv(r"your_directory\different_stimuli_samp_pol_del_gc_ASH-AVA_ASH-AVB_adult.csv")  # [ CHANGE NAME ]

---

In [ ]:
# (modified from supporting function 'plot_reverse_probability_per_input') ====================================================================

def plot_reverse_probability_per_input_dauer(sensory_voltage_changes, responses_df_dauer_and_herm, title, anchor, *args):
    '''
    Inputs:
        1. sensory_voltage_changes - list of voltage changes to the sensory neuron that were tested.
        2. responses_df_dauer_and_herm - df with the WT responses of both stages
        3. title
        4. anchor - for legend
    args:
    optional. for connectivity perturbations, it is possible to add additional curves with the 
    responses of the masculinized\feminized worms. 
        1. dimorphic_df_change (args[i * 4])
        2. dimorphic_change (args[i * 4 + 1])
        3. label (args[i * 4 + 2])
        4. additional_color (args[i * 4 + 3])
        5. if args > 4, repeat 1-4. 
    
    Output:
       plot (saved and presented) 
    '''
    mean_reversal_herm = []
    mean_reversal_dauer = []
    for sens_voltage in range(len(sensory_voltage_changes)):
        mean_reversal_herm.append(np.mean(responses_df_dauer_and_herm["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
        mean_reversal_dauer.append(np.mean(responses_df_dauer_and_herm["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    plot_margins = 2    
    if len(args)>0:
        dimorphic_df_change = []
        dimorphic_changes = []
        labels = []
        additional_colors =[]
        for i in range(int(len(args)/4)):
            dimorphic_df_change.append(args[i * 4])
            dimorphic_changes.append(args[i * 4 + 1])
            labels.append(args[i * 4 + 2])
            additional_colors.append(args[i * 4 + 3])
            mean_reversal_changed_con = {}
        for change in range(len(dimorphic_changes)):
            mean_reversal_changed_con[labels[change]] = []
            for sens_voltage in range(len(sensory_voltage_changes)):
                mean_reversal_changed_con[labels[change]] += [np.mean(dimorphic_df_change[change]["A_vs_B_"+ dimorphic_changes[change] + "_" +str(sensory_voltage_changes[sens_voltage])+"mV_change"])]
            plt.plot(sensory_voltage_changes,mean_reversal_changed_con[labels[change]], marker = "o", label = labels[change], color = additional_colors[change])
    
    plt.plot(sensory_voltage_changes,mean_reversal_herm, marker = "o", label = "Hermaphrodites", color = 'sandybrown')
    plt.plot(sensory_voltage_changes,mean_reversal_dauer, marker = "o", label = "Dauers", color ='tomato')
    plt.xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
    plt.ylabel("Predicted Avoidance", fontsize = 16)
    plt.xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=16)
    if len(args)>0:
        plt.legend(labels + ["Wild-type hermaphrodites","Wild-type dauers"] , fontsize = 14, loc='lower right', bbox_to_anchor = anchor)
    else:
        plt.legend(["Wild-type hermaphrodites","Wild-type dauers"] , fontsize = 14, loc='lower right', bbox_to_anchor = anchor)
    
    if "\\" in title:
        title_on_plot = title.split("\\")[-1]
    else: title_on_plot == title
    
    plt.title(title_on_plot, fontsize = 18)
    plt.savefig(title +".png", dpi=400, facecolor='w', bbox_inches='tight')
    plt.show()

# =============================================================================================================================================

In [ ]:
# (modified from supporting function 'plot_reverse_probability_per_input') ====================================================================

def plot_reverse_probability_per_input_all(sensory_voltage_changes, responses_df_all, title, anchor, *args):
    '''
    Inputs:
        1. sensory_voltage_changes - list of voltage changes to the sensory neuron that were tested.
        2. responses_df_all - df with the WT responses of hermaphrodite, male, and dauer
        3. title
        4. anchor - for legend
    args:
    optional. for connectivity perturbations, it is possible to add additional curves with the 
    responses of the masculinized\feminized worms. 
        1. dimorphic_df_change (args[i * 4])
        2. dimorphic_change (args[i * 4 + 1])
        3. label (args[i * 4 + 2])
        4. additional_color (args[i * 4 + 3])
        5. if args > 4, repeat 1-4. 
    
    Output:
       plot (saved and presented) 
    '''
    mean_reversal_herm = []
    mean_reversal_male = []
    mean_reversal_dauer = []
    for sens_voltage in range(len(sensory_voltage_changes)):
        mean_reversal_herm.append(np.mean(responses_df_all["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
        mean_reversal_male.append(np.mean(responses_df_all["male_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
        mean_reversal_dauer.append(np.mean(responses_df_all["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    plot_margins = 2    
    if len(args)>0:
        dimorphic_df_change = []
        dimorphic_changes = []
        labels = []
        additional_colors =[]
        for i in range(int(len(args)/4)):
            dimorphic_df_change.append(args[i * 4])
            dimorphic_changes.append(args[i * 4 + 1])
            labels.append(args[i * 4 + 2])
            additional_colors.append(args[i * 4 + 3])
            mean_reversal_changed_con = {}
        for change in range(len(dimorphic_changes)):
            mean_reversal_changed_con[labels[change]] = []
            for sens_voltage in range(len(sensory_voltage_changes)):
                mean_reversal_changed_con[labels[change]] += [np.mean(dimorphic_df_change[change]["A_vs_B_"+ dimorphic_changes[change] + "_" +str(sensory_voltage_changes[sens_voltage])+"mV_change"])]
            plt.plot(sensory_voltage_changes,mean_reversal_changed_con[labels[change]], marker = "o", label = labels[change], color = additional_colors[change])
    
    plt.plot(sensory_voltage_changes,mean_reversal_herm, marker = "o", label = "Hermaphrodites", color = 'sandybrown')  # orangish
    plt.plot(sensory_voltage_changes,mean_reversal_male, marker = "o", label = "Males", color = 'darkturquoise')  # bluish
    plt.plot(sensory_voltage_changes,mean_reversal_dauer, marker = "o", label = "Dauers", color ='tomato')  # redish
    plt.xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
    plt.ylabel("Predicted Avoidance", fontsize = 16)
    plt.xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=16)
    if len(args)>0:
        plt.legend(labels + ["Wild-type hermaphrodites","Wild-type males", "Wild-type dauers"] , fontsize = 14, loc='upper right', bbox_to_anchor = anchor)
    else:
        plt.legend(["Wild-type hermaphrodites","Wild-type males", "Wild-type dauers"] , fontsize = 14, loc='upper right', bbox_to_anchor = anchor)
    
    if "\\" in title:
        title_on_plot = title.split("\\")[-1]
    else: title_on_plot == title
    
    plt.title(title_on_plot, fontsize = 18)
    plt.savefig(title +".png", dpi=400, facecolor='w', bbox_inches='tight')
    plt.show()

# =============================================================================================================================================

---

In [ ]:
# plot

from os import path
import seaborn as sns
import supporting_functions as sf


different_stimuli_samp_pol = pd.read_csv(r"your_directory\different_stimuli_samp_pol_del_g_ASH-AVA_c_ASH-AVB.csv")  # [ CHANGE ]

title_1 = r"saved_graphs\\different_sensory_stimuli"

# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]


# plot flip (modified function)
plot_reverse_probability_per_input_dauer(sensory_voltage_changes, different_stimuli_samp_pol,
                                         r"your_directory\predicted_avoidance_dauer_del_g_ASH-AVA_c_ASH-AVB", (0.99,0.01))  # [ CHANGE ]


---

In [ ]:
# Data

different_stimuli_samp_pol_ori = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667.csv")
    # original (hermaphrodite adult, male, dauer)

different_stimuli_samp_pol_cfold0 = pd.read_csv(r"your_directory\flip\different_stimuli_samp_pol_flip_cfold_idx0.csv")
different_stimuli_samp_pol_cfold1 = pd.read_csv(r"your_directory\flip\different_stimuli_samp_pol_flip_cfold_idx1.csv")
different_stimuli_samp_pol_cfold2 = pd.read_csv(r"your_directory\flip\different_stimuli_samp_pol_flip_cfold_idx2.csv")
different_stimuli_samp_pol_cfold3 = pd.read_csv(r"your_directory\flip\different_stimuli_samp_pol_flip_cfold_idx3.csv")
different_stimuli_samp_pol_cfold4 = pd.read_csv(r"your_directory\flip\different_stimuli_samp_pol_flip_cfold_idx4.csv")

different_stimuli_samp_pol_cbi0 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_flip_cbi_idx0.csv")
different_stimuli_samp_pol_cbi1 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_flip_cbi_idx1.csv")
different_stimuli_samp_pol_cbi2 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_flip_cbi_idx2.csv")

different_stimuli_samp_pol_gfold0 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_flip_gfold_idx0.csv")
different_stimuli_samp_pol_gfold2 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_flip_gfold_idx2.csv")
different_stimuli_samp_pol_gfold3 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_flip_gfold_idx3.csv")
different_stimuli_samp_pol_gfold4 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_flip_gfold_idx4.csv")
different_stimuli_samp_pol_gfold6 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_flip_gfold_idx6.csv")

different_stimuli_samp_pol_gbi0 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_flip_gbi_idx0.csv")
different_stimuli_samp_pol_gbi2 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_flip_gbi_idx2.csv")
different_stimuli_samp_pol_gbi4 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_flip_gbi_idx4.csv")


In [ ]:
# plot 
    # All 'FLIP' cases + original 'adult' & 'dauer'


from os import path
import seaborn as sns
import supporting_functions as sf


# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]


mean_reversal_herm = []
mean_reversal_male = []
mean_reversal_dauer = []

mean_reversal_cfold0 = []; mean_reversal_cfold1 = []; mean_reversal_cfold2 = []; mean_reversal_cfold3 = []; mean_reversal_cfold4 = []
mean_reversal_gfold0 = []; mean_reversal_gfold2 = []; mean_reversal_gfold3 = []; mean_reversal_gfold4 = []; mean_reversal_gfold6 = []

mean_reversal_cbi0 = []; mean_reversal_cbi1 = []; mean_reversal_cbi2 = []
mean_reversal_gbi0 = []; mean_reversal_gbi2 = []; mean_reversal_gbi4 = []


for sens_voltage in range(len(sensory_voltage_changes)):
    # original
    mean_reversal_herm.append(np.mean(different_stimuli_samp_pol_ori["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_male.append(np.mean(different_stimuli_samp_pol_ori["male_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_dauer.append(np.mean(different_stimuli_samp_pol_ori["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    # fold
    mean_reversal_cfold0.append(np.mean(different_stimuli_samp_pol_cfold0["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_cfold1.append(np.mean(different_stimuli_samp_pol_cfold1["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_cfold2.append(np.mean(different_stimuli_samp_pol_cfold2["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_cfold3.append(np.mean(different_stimuli_samp_pol_cfold3["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_cfold4.append(np.mean(different_stimuli_samp_pol_cfold4["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_gfold0.append(np.mean(different_stimuli_samp_pol_gfold0["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_gfold2.append(np.mean(different_stimuli_samp_pol_gfold2["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_gfold3.append(np.mean(different_stimuli_samp_pol_gfold3["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_gfold4.append(np.mean(different_stimuli_samp_pol_gfold4["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_gfold6.append(np.mean(different_stimuli_samp_pol_gfold6["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    # binary
    mean_reversal_cbi0.append(np.mean(different_stimuli_samp_pol_cbi0["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_cbi1.append(np.mean(different_stimuli_samp_pol_cbi1["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_cbi2.append(np.mean(different_stimuli_samp_pol_cbi2["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_gbi0.append(np.mean(different_stimuli_samp_pol_gbi0["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_gbi2.append(np.mean(different_stimuli_samp_pol_gbi2["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_gbi4.append(np.mean(different_stimuli_samp_pol_gbi4["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    
plot_margins = 2


fig, ((ax1)) = plt.subplots(1, 1, sharex=False, sharey=False, figsize=(11,8))

# original
ax1.plot(sensory_voltage_changes,mean_reversal_herm, marker = "o", label = "Hermaphrodites", color = 'sandybrown', lw=4)  # orangish
ax1.plot(sensory_voltage_changes,mean_reversal_male, marker = "o", label = "Males", color = 'darkturquoise', lw=4)  # bluish
ax1.plot(sensory_voltage_changes,mean_reversal_dauer, marker = "o", label = "Dauers", color ='tomato', lw=4)  # redish
# chemical (brownish)
ax1.plot(sensory_voltage_changes,mean_reversal_cfold0, marker = ".", label = "CHEM fold 0", color ='papayawhip', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_cfold1, marker = ".", label = "CHEM fold 1", color ='blanchedalmond', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_cfold2, marker = ".", label = "CHEM fold 2", color ='moccasin', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_cfold3, marker = ".", label = "CHEM fold 3", color ='navajowhite', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_cfold4, marker = ".", label = "CHEM fold 4", color ='wheat', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_cbi0, marker = ".", label = "CHEM bi 0", color ='gold', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_cbi1, marker = ".", label = "CHEM bi 1", color ='goldenrod', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_cbi2, marker = ".", label = "CHEM bi 2", color ='darkgoldenrod', ls="dashed")
# gap (bluish)
ax1.plot(sensory_voltage_changes,mean_reversal_gfold0, marker = ".", label = "GAP fold 0", color ='powderblue', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_gfold2, marker = ".", label = "GAP fold 2", color ='lightblue', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_gfold3, marker = ".", label = "GAP fold 3", color ='skyblue', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_gfold4, marker = ".", label = "GAP fold 4", color ='lightskyblue', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_gfold6, marker = ".", label = "GAP fold 6", color ='lightsteelblue', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_gbi0, marker = ".", label = "GAP bi 0", color ='deepskyblue', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_gbi2, marker = ".", label = "GAP bi 2", color ='cornflowerblue', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_gbi4, marker = ".", label = "GAP bi 4", color ='royalblue', ls="dashed")

ax1.set_xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
ax1.set_ylabel("Predicted Avoidance", fontsize = 16)
ax1.set_xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=16)
ax1.legend(loc='upper left', bbox_to_anchor=(1.02,1), fontsize=14)
ax1.set_title("Dauer dimorphic connection FLIP", fontsize = 21)

# plt.savefig(r"your_directory\predicted_avoidance_dauer_FLIP_all.svg", dpi=300, facecolor='w', bbox_inches='tight', pad_inches=0)
plt.show()


In [ ]:
# plot (Chemical flip only)


from os import path
import seaborn as sns
import supporting_functions as sf


# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]


mean_reversal_herm = []
mean_reversal_dauer = []

mean_reversal_cfold0 = []; mean_reversal_cfold1 = []; mean_reversal_cfold2 = []; mean_reversal_cfold3 = []; mean_reversal_cfold4 = []
mean_reversal_cbi0 = []; mean_reversal_cbi1 = []; mean_reversal_cbi2 = []


for sens_voltage in range(len(sensory_voltage_changes)):
    # original
    mean_reversal_herm.append(np.mean(different_stimuli_samp_pol_ori["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_dauer.append(np.mean(different_stimuli_samp_pol_ori["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    # fold
    mean_reversal_cfold0.append(np.mean(different_stimuli_samp_pol_cfold0["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_cfold1.append(np.mean(different_stimuli_samp_pol_cfold1["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_cfold2.append(np.mean(different_stimuli_samp_pol_cfold2["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_cfold3.append(np.mean(different_stimuli_samp_pol_cfold3["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_cfold4.append(np.mean(different_stimuli_samp_pol_cfold4["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    # binary
    mean_reversal_cbi0.append(np.mean(different_stimuli_samp_pol_cbi0["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_cbi1.append(np.mean(different_stimuli_samp_pol_cbi1["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_cbi2.append(np.mean(different_stimuli_samp_pol_cbi2["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    
plot_margins = 2


fig, ((ax1)) = plt.subplots(1, 1, sharex=False, sharey=False, figsize=(11,8))

# original
ax1.plot(sensory_voltage_changes,mean_reversal_herm, marker = "o", label = "Hermaphrodites", color = 'sandybrown', lw=4)  # orangish
ax1.plot(sensory_voltage_changes,mean_reversal_dauer, marker = "o", label = "Dauers", color ='tomato', lw=4)  # redish
# chemical (brownish)
ax1.plot(sensory_voltage_changes,mean_reversal_cfold0, marker = ".", label = "AVB → AVA", color ='papayawhip', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_cfold1, marker = ".", label = "PVC → AVA", color ='blanchedalmond', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_cfold2, marker = ".", label = "PVC → AVD", color ='moccasin', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_cfold3, marker = ".", label = "ASH → AVB", color ='navajowhite', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_cfold4, marker = ".", label = "PVC → AVB", color ='wheat', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_cbi0, marker = ".", label = "AVD → AVA", color ='gold', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_cbi1, marker = ".", label = "AVA → AVD", color ='goldenrod', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_cbi2, marker = ".", label = "AVA → PVC", color ='darkgoldenrod', ls="dashed")

ax1.set_xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
ax1.set_ylabel("Predicted Avoidance", fontsize = 16)
ax1.set_xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=16)
ax1.legend(loc='lower right', bbox_to_anchor=(1,0), fontsize=14)
ax1.set_title("Dauer dimorphic connection FLIP (Chemical)", fontsize = 21)

# plt.savefig(r"your_directory\predicted_avoidance_dauer_FLIP_CHEM.svg", dpi=300, facecolor='w', bbox_inches='tight', pad_inches=0)
plt.show()


In [ ]:
# plot (Gap flip only)


from os import path
import seaborn as sns
import supporting_functions as sf


# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]


mean_reversal_herm = []
mean_reversal_dauer = []

mean_reversal_gfold0 = []; mean_reversal_gfold2 = []; mean_reversal_gfold3 = []; mean_reversal_gfold4 = []; mean_reversal_gfold6 = []
mean_reversal_gbi0 = []; mean_reversal_gbi2 = []; mean_reversal_gbi4 = []


for sens_voltage in range(len(sensory_voltage_changes)):
    # original
    mean_reversal_herm.append(np.mean(different_stimuli_samp_pol_ori["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_dauer.append(np.mean(different_stimuli_samp_pol_ori["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    # fold
    mean_reversal_gfold0.append(np.mean(different_stimuli_samp_pol_gfold0["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_gfold2.append(np.mean(different_stimuli_samp_pol_gfold2["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_gfold3.append(np.mean(different_stimuli_samp_pol_gfold3["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_gfold4.append(np.mean(different_stimuli_samp_pol_gfold4["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_gfold6.append(np.mean(different_stimuli_samp_pol_gfold6["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    # binary
    mean_reversal_gbi0.append(np.mean(different_stimuli_samp_pol_gbi0["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_gbi2.append(np.mean(different_stimuli_samp_pol_gbi2["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_gbi4.append(np.mean(different_stimuli_samp_pol_gbi4["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    
plot_margins = 2


fig, ((ax1)) = plt.subplots(1, 1, sharex=False, sharey=False, figsize=(11,8))

# original
ax1.plot(sensory_voltage_changes,mean_reversal_herm, marker = "o", label = "Hermaphrodites", color = 'sandybrown', lw=4)  # orangish
ax1.plot(sensory_voltage_changes,mean_reversal_dauer, marker = "o", label = "Dauers", color ='tomato', lw=4)  # redish
# gap (bluish)
ax1.plot(sensory_voltage_changes,mean_reversal_gfold0, marker = ".", label = "ASH = AVA", color ='powderblue', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_gfold2, marker = ".", label = "AVA = AVD", color ='lightblue', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_gfold3, marker = ".", label = "AVA = AVB", color ='skyblue', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_gfold4, marker = ".", label = "AVA = PVC", color ='lightskyblue', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_gfold6, marker = ".", label = "AVD = PVC", color ='lightsteelblue', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_gbi0, marker = ".", label = "ASH = AVD", color ='deepskyblue', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_gbi2, marker = ".", label = "AVD = AVB", color ='cornflowerblue', ls="dashed")
ax1.plot(sensory_voltage_changes,mean_reversal_gbi4, marker = ".", label = "AVB = PVC", color ='royalblue', ls="dashed")

ax1.set_xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
ax1.set_ylabel("Predicted Avoidance", fontsize = 16)
ax1.set_xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=16)
ax1.legend(loc='lower right', bbox_to_anchor=(1,0), fontsize=14)
ax1.set_title("Dauer dimorphic connection FLIP (Gap)", fontsize = 21)

# plt.savefig(r"your_directory\predicted_avoidance_dauer_FLIP_GAP.svg", dpi=300, facecolor='w', bbox_inches='tight', pad_inches=0)
plt.show()


In [ ]:
# ASH-AVA delete

different_stimuli_samp_pol_ori = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667.csv")
    # original (hermaphrodite adult, male, dauer)

different_stimuli_samp_pol_gc_ASH_AVA_dauer = pd.read_csv(r"your_directory\different_stimuli_samp_pol_flip_gc_ASH-AVA.csv")


# plot 
    # gfold03 cases + original 'adult' & 'dauer'

# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]


mean_reversal_herm = []
mean_reversal_male = []
mean_reversal_dauer = []

mean_reversal_gc_ASH_AVA_dauer = []


for sens_voltage in range(len(sensory_voltage_changes)):
    # original
    mean_reversal_herm.append(np.mean(different_stimuli_samp_pol_ori["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_male.append(np.mean(different_stimuli_samp_pol_ori["male_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_dauer.append(np.mean(different_stimuli_samp_pol_ori["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    # fold
    mean_reversal_gc_ASH_AVA_dauer.append(np.mean(different_stimuli_samp_pol_gc_ASH_AVA_dauer["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    
plot_margins = 2


fig, ((ax1)) = plt.subplots(1, 1, sharex=False, sharey=False, figsize=(8,5))

# original
ax1.plot(sensory_voltage_changes,mean_reversal_herm, marker = "o", label = "Hermaphrodites", color = 'sandybrown', lw=4)  # orangish
ax1.plot(sensory_voltage_changes,mean_reversal_male, marker = "o", label = "Males", color = 'darkturquoise', lw=4)  # bluish
ax1.plot(sensory_voltage_changes,mean_reversal_dauer, marker = "o", label = "Dauers", color ='tomato', lw=4)  # redish
# gap (bluish)
ax1.plot(sensory_voltage_changes,mean_reversal_gc_ASH_AVA_dauer, marker = ".", label = "ASH==AVA & ASH->AVA (DEL)", color ='royalblue', ls="dashed", lw=2.5)

ax1.set_xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
ax1.set_ylabel("Predicted Avoidance", fontsize = 16)
ax1.set_xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=16)
ax1.legend(loc='upper left', bbox_to_anchor=(1.02,1), fontsize=14)
ax1.set_title("No connectivity between ASH & AVA", fontsize = 21)

# plt.savefig(r"your_directory\predicted_avoidance_dauer_gc_ASH-AVA.png", dpi=400, facecolor='w', bbox_inches='tight')
plt.show()


In [ ]:
# ASH-AVA delete (in adult)

different_stimuli_samp_pol_ori = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667.csv")
    # original (hermaphrodite adult, male, dauer)

different_stimuli_samp_pol_gc_ASH_AVA_adult = pd.read_csv(r"your_directory\different_stimuli_samp_pol_flip_gc_ASH-AVA_adult.csv")


# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]


mean_reversal_herm = []
mean_reversal_male = []
mean_reversal_dauer = []

mean_reversal_gc_ASH_AVA_adult = []


for sens_voltage in range(len(sensory_voltage_changes)):
    # original
    mean_reversal_herm.append(np.mean(different_stimuli_samp_pol_ori["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_male.append(np.mean(different_stimuli_samp_pol_ori["male_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_dauer.append(np.mean(different_stimuli_samp_pol_ori["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    # fold
    mean_reversal_gc_ASH_AVA_adult.append(np.mean(different_stimuli_samp_pol_gc_ASH_AVA_adult["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
        # Note that "herm_A_vs_B_" is changed as well
    
plot_margins = 2


fig, ((ax1)) = plt.subplots(1, 1, sharex=False, sharey=False, figsize=(8,5))

# original
ax1.plot(sensory_voltage_changes,mean_reversal_herm, marker = "o", label = "Hermaphrodites", color = 'sandybrown', lw=4)  # orangish
ax1.plot(sensory_voltage_changes,mean_reversal_male, marker = "o", label = "Males", color = 'darkturquoise', lw=4)  # bluish
ax1.plot(sensory_voltage_changes,mean_reversal_dauer, marker = "o", label = "Dauers", color ='tomato', lw=4)  # redish
# gap (bluish)
ax1.plot(sensory_voltage_changes,mean_reversal_gc_ASH_AVA_adult, marker = ".", label = "ASH==AVA & ASH->AVA (DEL; adult)", color ='royalblue', ls="dashed", lw=2.5)

ax1.set_xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
ax1.set_ylabel("Predicted Avoidance", fontsize = 16)
ax1.set_xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=16)
ax1.legend(loc='upper left', bbox_to_anchor=(1.02,1), fontsize=14)
ax1.set_title("No connectivity between ASH & AVA (adult)", fontsize = 21)

# plt.savefig(r"your_directory\predicted_avoidance_dauer_gc_ASH-AVA_adult.png", dpi=400, facecolor='w', bbox_inches='tight')
plt.show()


In [ ]:
# ASH-AVA delete ; all possible combination

    ##   ASH->AVA CHEM = 1   ##
    ##   ASH==AVA  GAP = 2   ##
    ##   ASH->AVB CHEM = 3   ##
    
    ##   express like ooo, xoo etc.


different_stimuli_samp_pol_ooo = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667.csv")  # ooo
    # original (hermaphrodite adult, male, dauer)

different_stimuli_samp_pol_xoo = pd.read_csv(r"your_directory\flip\different_stimuli_samp_pol_del_c_ASH-AVA.csv")  # xoo
different_stimuli_samp_pol_oxo = pd.read_csv(r"your_directory\flip\different_stimuli_samp_pol_del_g_ASH-AVA.csv")  # oxo
different_stimuli_samp_pol_xxo = pd.read_csv(r"your_directory\flip\different_stimuli_samp_pol_flip_gc_ASH-AVA.csv")  # xxo



# plot 

# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]


mean_reversal_ooo_herm = []
mean_reversal_ooo_male = []
mean_reversal_ooo_dauer = []

mean_reversal_xoo_dauer = []
mean_reversal_oxo_dauer = []
mean_reversal_xxo_dauer = []


for sens_voltage in range(len(sensory_voltage_changes)):
    # original
    mean_reversal_ooo_herm.append(np.mean(different_stimuli_samp_pol_ooo["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_ooo_male.append(np.mean(different_stimuli_samp_pol_ooo["male_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_ooo_dauer.append(np.mean(different_stimuli_samp_pol_ooo["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    # delete
    mean_reversal_xoo_dauer.append(np.mean(different_stimuli_samp_pol_xoo["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_oxo_dauer.append(np.mean(different_stimuli_samp_pol_oxo["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_xxo_dauer.append(np.mean(different_stimuli_samp_pol_xxo["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    
    
plot_margins = 2


fig, ((ax1)) = plt.subplots(1, 1, sharex=False, sharey=False, figsize=(8,5))

# original
ax1.plot(sensory_voltage_changes,mean_reversal_ooo_herm, marker = "o", label = "Adults", color = 'sandybrown', lw=4)  # orangish
ax1.plot(sensory_voltage_changes,mean_reversal_ooo_male, marker = "o", label = "Males", color = 'darkturquoise', lw=4)  # bluish
ax1.plot(sensory_voltage_changes,mean_reversal_ooo_dauer, marker = "o", label = "Dauers", color ='tomato', lw=4)  # redish

# delete combination
ax1.plot(sensory_voltage_changes,mean_reversal_xoo_dauer, marker = ".", label = "ASH→AVA X , ASH=AVA O", color ='hotpink', ls="dashed", lw=2.5)
ax1.plot(sensory_voltage_changes,mean_reversal_oxo_dauer, marker = ".", label = "ASH→AVA O , ASH=AVA X", color ='deepskyblue', ls="dashed", lw=2.5)
ax1.plot(sensory_voltage_changes,mean_reversal_xxo_dauer, marker = ".", label = "ASH→AVA X , ASH=AVA X", color ='royalblue', ls="dashed", lw=2.5)


ax1.set_xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
ax1.set_ylabel("Predicted Avoidance", fontsize = 16)
ax1.set_xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=16)
ax1.legend(loc='upper left', bbox_to_anchor=(1.02,1), fontsize=16)
ax1.set_title("ASH-AVA delete combination (Dauer)", fontsize = 21)

# plt.savefig(r"your_directory\predicted_avoidance_dauer_ASH-AVA_all_combination_del_260102.svg", dpi=300, facecolor='w', bbox_inches='tight', pad_inches=0)
plt.show()


In [ ]:
# ASH-AVA delete in Adult(herm); all possible combination
    # Pechuk Fig.6A replicate


different_stimuli_samp_pol_ori = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667.csv")  # original (hermaphrodite adult, male, dauer)

different_stimuli_samp_pol_adult_cdel = pd.read_csv(r"your_directory\adult\different_stimuli_samp_pol_del_c_ASH-AVA_adult.csv")  # chem delete
different_stimuli_samp_pol_adult_gdel = pd.read_csv(r"your_directory\adult\different_stimuli_samp_pol_del_g_ASH-AVA_adult.csv")  # gap delete
different_stimuli_samp_pol_adult_adel = pd.read_csv(r"your_directory\adult\different_stimuli_samp_pol_del_gc_ASH-AVA_adult.csv")  # all delete (gap + chem)



# plot 

# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]


mean_reversal_ori_herm = []
mean_reversal_ori_male = []
mean_reversal_ori_dauer = []

mean_reversal_cdel_herm = []
mean_reversal_gdel_herm = []
mean_reversal_adel_herm = []



for sens_voltage in range(len(sensory_voltage_changes)):
    # original
    mean_reversal_ori_herm.append(np.mean(different_stimuli_samp_pol_ori["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_ori_male.append(np.mean(different_stimuli_samp_pol_ori["male_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_ori_dauer.append(np.mean(different_stimuli_samp_pol_ori["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    # delete
    mean_reversal_cdel_herm.append(np.mean(different_stimuli_samp_pol_adult_cdel["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_gdel_herm.append(np.mean(different_stimuli_samp_pol_adult_gdel["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    mean_reversal_adel_herm.append(np.mean(different_stimuli_samp_pol_adult_adel["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    
plot_margins = 2


fig, ((ax1)) = plt.subplots(1, 1, sharex=False, sharey=False, figsize=(8,5))

# original
ax1.plot(sensory_voltage_changes,mean_reversal_ori_herm, marker = "o", label = "Adults", color = 'sandybrown', lw=4)  # orangish
ax1.plot(sensory_voltage_changes,mean_reversal_ori_male, marker = "o", label = "Males", color = 'darkturquoise', lw=4)  # bluish
ax1.plot(sensory_voltage_changes,mean_reversal_ori_dauer, marker = "o", label = "Dauers", color ='tomato', lw=4)  # redish

# delete combination
ax1.plot(sensory_voltage_changes,mean_reversal_cdel_herm, marker = ".", label = "ASH→AVA X , ASH=AVA O", color ='hotpink', ls="dashed", lw=2.5)
ax1.plot(sensory_voltage_changes,mean_reversal_gdel_herm, marker = ".", label = "ASH→AVA O , ASH=AVA X", color ='deepskyblue', ls="dashed", lw=2.5)
ax1.plot(sensory_voltage_changes,mean_reversal_adel_herm, marker = ".", label = "ASH→AVA X , ASH=AVA X", color ='royalblue', ls="dashed", lw=2.5)


ax1.set_xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
ax1.set_ylabel("Predicted Avoidance", fontsize = 16)
ax1.set_xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=16)
ax1.legend(loc='upper left', bbox_to_anchor=(1.02,1), fontsize=16)
ax1.set_title("ASH-AVA delete combination (Adult)", fontsize = 21)

# plt.savefig(r"your_directory\predicted_avoidance_adult_ASH-AVA_all_combination_del_Pechuk_6A_replicate_260102.svg", dpi=300, facecolor='w', bbox_inches='tight', pad_inches=0)
plt.show()


---
---